# Third-revision reproduction notebook

This notebook reproduces the second-round and third-revision analyses from the released data and code. It is light by design: it reads the released result files and checks the numbers quoted in the manuscript. The full runs (equal-work, calibration, timing, k = 6 emulation) take hours and are launched from the scripts in `code/revision2/` and `code/extensions/`. Nothing in this notebook runs a new simulation, except section 4 (a few seconds) and section 5 (optional, minutes).

Run all cells from the repository root.

**Changes in this version (third revision):** new section 1a (Bonferroni families and their sizes), new section 1b (k = 6 total demand, 4313.5 Mbps versus 1278.3 × 54/16), and new section 1c (weight-set analysis of the headline comparison: S2 only).

In [ ]:
import os, sys, json
import pandas as pd
ROOT = os.path.abspath('.')
sys.path.insert(0, os.path.join(ROOT, 'code', 'revision2'))
sys.path.insert(0, os.path.join(ROOT, 'code'))
print('root:', ROOT)

## 1. Paired comparisons (baseline and calibrated objective)

Paired t-test, 95% t-interval and Wilcoxon signed-rank for each comparison. Positive mean difference means the first-named method loses more flows. Bonferroni is over six comparisons per scenario (column `p_bonf_m6`, family (iv) in Section 3.7 of the manuscript).

In [ ]:
comp = pd.read_csv(os.path.join(ROOT, 'data', 'revision2', 'revision_comparisons.csv'))
comp

## 1a. Bonferroni families of the second round

The file holds family (iv): six comparisons for each of S1, S2 and S3. The calibrated-objective contrasts belong to this family only. The headline family of seven settings (Table 4) and the other families are not in this file.

In [ ]:
fam = comp.groupby('scenario').size()
print(fam.to_string())
assert len(comp) == 18 and (fam == 6).all()
cal = comp[comp['comparison'].str.contains('calibrated')]
print(cal[['scenario','comparison','mean_diff','ci_low','ci_high','p_t','p_bonf_m6']].round(4).to_string())
# values quoted in the manuscript (Section 4.2 and Section 5.2)
expected = {'S1': 0.0522, 'S3': 0.0864}
for s, p in expected.items():
    got = float(cal.loc[cal['scenario']==s, 'p_bonf_m6'].iloc[0])
    assert abs(got - p) < 5e-5, (s, got)
print('calibrated contrasts: S1 and S3 not significant after m = 6 correction')

## 1b. k = 6 scenario: total demand

The k = 6 runs use D_tot = 4313.5 Mbps (column `D` of the released result files). The exact scaling of the S2 demand, 1278.3 × 54/16, gives 4314.2625 Mbps. The generator rescales the drawn demands to sum to D_tot, so the difference is a uniform factor on every demand. This cell reads the released files only; the k = 6 runs themselves are not repeated here (about 6.5 min per SA seed).

In [ ]:
k6 = pd.read_csv(os.path.join(ROOT, 'code', 'extensions', 'results_k6_priority_emulation', 'k6_main.csv'), usecols=['D'])
D_used = sorted(float(x) for x in k6['D'].unique())
exact = 1278.3 * 54 / 16
print('D used in k6_main.csv:', D_used)
print('exact 1278.3 x 54/16 =', round(exact, 4), 'Mbps')
print('difference =', round(exact - 4313.5, 2), 'Mbps,', round((exact - 4313.5) / exact * 100, 3), '%')
print('uniform factor on every flow demand =', round(4313.5 / exact, 5))
print('demand per host: k=6', round(4313.5 / 54, 2), ' S2', round(1278.3 / 16, 2), 'Mbps')
assert D_used == [4313.5]

## 1c. Weight sets: scope of the headline comparison

The headline comparison (monolithic SQA minus tuned SA) was repeated under all six weight sets on S2 only (30 seeds; Section 4.2). In S1 and S3 and in the scale study it uses the baseline weights. The per-set statistics are in `resim_table_headline_lambda.csv`, which is in the Zenodo record (`results_main/`), not in this repository. The check below prints it when it is present locally.

In [ ]:
p = os.path.join(ROOT, 'data', 'revision2', 'resim_table_headline_lambda.csv')
if os.path.exists(p):
    t = pd.read_csv(p)
    print(t.to_string())
else:
    print('resim_table_headline_lambda.csv is not in this checkout; download results_main/ from Zenodo record 24152123 to run this check.')

## 2. Calibration validation (seeds 100–104, S2)

Mean PL_flow after repair for each admission bonus mu. mu* = 20 is the choice used in the manuscript.

In [ ]:
val = pd.read_csv(os.path.join(ROOT, 'data', 'revision2', 'e4a_mu_validation.csv'))
val.groupby('mu')[['PL_flow','energy_sample','energy_all_rejected']].mean().round(2)

## 3. End-to-end timing (S2, 30 seeds, run alone)

Solve and coordinator time are measured separately. Pod time is the sum of the four sequential pod solves. These are single-machine measurements on a two-core sandbox.

In [ ]:
summ = pd.read_csv(os.path.join(ROOT, 'data', 'revision2', 'e23_summary.csv'))
summ

## 4. Software check: release pipeline on S1, seed 42

The release `run_mas` must reproduce the archived MAS-QA result for seed 42 (PL_flow = 26.0, BWL = 32.9926). This takes a few seconds.

In [ ]:
import masqa_lib as L
import r2lib as R
flows = L.gen_traffic(50, 640.0, 42)
cc = R.cc_of(flows)
served = L.run_mas(flows, 42, 640.0, cc)
served = served[0] if isinstance(served, tuple) else served
m = L.metrics(flows, served, 640.0)
print('PL_flow =', m['PL_flow'], ' BWL =', round(m['BWL'], 4))
assert m['PL_flow'] == 26.0

## 5. Energy check: exhaustive enumeration of a small QUBO

The 3-flow, 2-arc instance has 19 binary variables (3 admission, 4 path, 12 slack), so 2^19 = 524,288 states. The script checks that the global minimum is the all-rejected state. It needs a few minutes and about 100 MB of memory. The stored result is in `data/revision2/d_enum.json`.

In [ ]:
print(open(os.path.join(ROOT, 'data', 'revision2', 'd_enum.json')).read())
# To recompute: %run code/revision2/d_enum.py